# 03 — Boundary Slices and Path Composition
## The Geometry of Chained Workflows, the Gluing Law, and the Evaluator Residual

**Series:** `finite_obstruction_calculus` / `012_lggr_i_evaluated_path_kernels`  
**Author:** Jeremy H. Carroll  
**Status:** Living Educational Supplement  
**Anchor Tag:** `sealed-derivation-v1` (`3100c4e`, cut 2026-08-17)

---

> ### **Claim-Typing Legend**
> - **[Pinned]**: Verified directly by executable Python functions in `code/epk_core.py` and passing test suites.
> - **[Proved]**: Mechanized in Lean 4 under `PaperAE004` (reported kernel axioms: `propext`, `Classical.choice`, `Quot.sound` or axiom-free).
> - **[Declared]**: An operational definition, architectural parameter, or illustrative teaching model chosen by the author.
> - **[Open]**: A mathematical or empirical question not discharged in this package.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. Workflows as Trajectories in State Space

When an autonomous agent executes a multi-step task, we can represent its progress as a **trajectory** through a finite state space:
- **$S_a$:** The initial state (e.g. user prompt, repo at `main`).
- **$S_b$:** An intermediate checkpoint or milestone (e.g. database schema updated, PR opened).
- **$S_c$:** The final terminal state (e.g. application deployed, pull request merged).

A trajectory $\gamma_1$ takes the system from $S_a \to S_b$, and a subsequent trajectory $\gamma_2$ takes the system from $S_b \to S_c$.
Their concatenation is the full end-to-end execution:
$$\gamma = \gamma_2 \star \gamma_1$$

### What is a Boundary Slice ($B$)?
In complex pipelines, there is rarely just one intermediate state. There may be multiple valid architectural patterns or microservice states.
A **Boundary Slice** $\mathsf{B} \subset \mathsf{S}$ is a declared cut across the workflow: **every valid full execution must pass through exactly one state $S_b \in \mathsf{B}$**.


## 2. The Three Structural Conditions: G1, G2, G3 (§3)

When can we evaluate sub-agents independently and multiply their success measures together to get the end-to-end success measure?

In Section 3, we derive that **three structural conditions are sufficient** for exact kernel composition:

1. **Condition G1 (Boundary Sufficiency & Unique Decomposition):**  
   Every composed path $\gamma \in \Omega(S_a, S_c)$ decomposes uniquely into a pair of segments $(\gamma_1, \gamma_2)$ through some state $S_b \in \mathsf{B}$, and all declared Cartesian segment combinations exist:
   $$\Omega(S_a, S_c) \cong \bigsqcup_{S_b \in \mathsf{B}} \Omega(S_b, S_c) \times \Omega(S_a, S_b)$$

2. **Condition G2 (Action Additivity):**  
   The execution cost or cognitive action of the composed path equals the sum of the segment actions:
   $$\mathcal{A}[\gamma_2 \star \gamma_1] = \mathcal{A}[\gamma_1] + \mathcal{A}[\gamma_2]$$

3. **Condition G3 (Evaluator Factorization):**  
   The global evaluator agrees with the conjunction of the local evaluators:
   $$E_g(\gamma_2 \star \gamma_1) = E_1(\gamma_1) \cdot E_2(\gamma_2) \quad \iff \quad \Delta_E = 0$$


## 3. What is an Evaluated Path Kernel? (§2)

To evaluate an agent system over declared recorded paths, we weight each path by its cost and filter it by its verification verdict:

$$Z_\beta(S_b \mid S_a) = \sum_{\gamma \in \Omega(S_a, S_b)} E(\gamma) \cdot e^{-\beta \mathcal{A}[\gamma]}$$

### Why the Exponential Weight $e^{-\beta \mathcal{A}}$?
- $\mathcal{A}[\gamma] \ge 0$ is the execution cost (e.g. latency, token count, API fee).
- $\beta \ge 0$ is the cost penalty parameter (inverse temperature).
  - If $\beta = 0$, every passing path counts equally ($e^0 = 1$), and $Z$ simply counts passing paths.
  - As $\beta > 0$ increases, low-cost efficient paths dominate the mass, while bloated, high-cost paths are exponentially penalized.
- $E(\gamma) \in \{0, 1\}$ is the hard evaluator: failing paths ($E = 0$) contribute zero mass!


In [ ]:
# Set up a workflow with 2 boundary states: B = {Sb1, Sb2}
s_a = State("Sa_Init")
s_b1 = State("Sb1_CheckpointA")
s_b2 = State("Sb2_CheckpointB")
s_c = State("Sc_Goal")

b_slice = BoundarySlice(states={s_b1, s_b2})

# Stage 1 segments (Sa -> Sb)
seg1_b1 = TrajectorySegment(s_a, s_b1, ("step1_fast",), raw_action=0.4, proposal_prob=0.6)
seg1_b2 = TrajectorySegment(s_a, s_b2, ("step1_deep",), raw_action=0.6, proposal_prob=0.4)

# Stage 2 segments (Sb -> Sc)
seg2_b1 = TrajectorySegment(s_b1, s_c, ("step2_standard",), raw_action=0.5, proposal_prob=0.7)
seg2_b2 = TrajectorySegment(s_b2, s_c, ("step2_heavy",), raw_action=0.8, proposal_prob=0.3)

# Composed trajectories
comp_b1 = ComposedTrajectory(seg1_b1, seg2_b1)
comp_b2 = ComposedTrajectory(seg1_b2, seg2_b2)

# Check G1: Verify unique decomposition through B
valid_g1, violations = b_slice.validate_unique_decomposition(
    [comp_b1, comp_b2],
    left_segments=[seg1_b1, seg1_b2],
    right_segments=[seg2_b1, seg2_b2]
)
print(f"Condition G1 (Boundary Decomposition Valid): {valid_g1} (Violations: {len(violations)})")
assert valid_g1

# Check G3: Factored evaluators (all pass)
eval_set_factored = EvaluatorSet(
    eval_1=lambda seg: 1,
    eval_2=lambda seg: 1,
    eval_g=lambda comp: 1
)

# Compute Stage 1 action kernels: Z(Sb | Sa)
z_b1_a = EvaluatedPathKernel.compute_action_kernel([seg1_b1], eval_set_factored.eval_1, beta=1.0)
z_b2_a = EvaluatedPathKernel.compute_action_kernel([seg1_b2], eval_set_factored.eval_1, beta=1.0)

# Compute Stage 2 action kernels: Z(Sc | Sb)
z_c_b1 = EvaluatedPathKernel.compute_action_kernel([seg2_b1], eval_set_factored.eval_2, beta=1.0)
z_c_b2 = EvaluatedPathKernel.compute_action_kernel([seg2_b2], eval_set_factored.eval_2, beta=1.0)

# Composed sum over boundary slice B: Z_composed = sum_{Sb in B} Z(Sc | Sb) * Z(Sb | Sa)
z_composed_sum = (z_c_b1 * z_b1_a) + (z_c_b2 * z_b2_a)

# Direct kernel over composed paths: sum exp(-beta * (A1 + A2))
# Path 1: 0.4 + 0.5 = 0.9 -> exp(-0.9)
# Path 2: 0.6 + 0.8 = 1.4 -> exp(-1.4)
z_direct = float(np.exp(-0.9) + np.exp(-1.4))

print(f"Direct Kernel Mass (Z_direct)       : {z_direct:.8f}")
print(f"Composed Kernel Mass (Z_composed)   : {z_composed_sum:.8f}")
rke, nrke = EvaluatedPathKernel.evaluate_composition(z_direct, z_composed_sum)
print(f"Residual Kernel Error (RKE)         : {rke:.1e}")

assert abs(rke) < 1e-15
print("\n[VERIFIED ON FIXTURE] Kernel Composition Theorem holds exactly under G1-G3!")


## 4. What Happens When G3 Fails? The Evaluator Residual (§4)

In real pipelines, G3 often fails: a path passes locally ($E_1=1, E_2=1$), but fails globally ($E_g=0$).

When G3 fails, composition is no longer exact. It acquires an exact residual error called the **Evaluator Residual** $R_{\mathrm{Eval}}$:

$$Z_\beta(S_c \mid S_a) - \sum_{S_b \in \mathsf{B}} Z_\beta(S_c \mid S_b) Z_\beta(S_b \mid S_a) = R_{\mathrm{Eval}}(S_c, S_a)$$

In Section 4, we prove the exact residual formula:
$$R_{\mathrm{Eval}}(S_c, S_a) = \sum_{\gamma \in \Omega(S_a, S_c)} \Delta_E(\gamma) \cdot e^{-\beta \mathcal{A}[\gamma]}$$

Let's test this directly:


In [ ]:
# Break G3 on Path b1: Local checks pass (E1=1, E2=1), but Global fails (Eg=0) -> Delta_E = -1
eval_set_broken = EvaluatorSet(
    eval_1=lambda seg: 1,
    eval_2=lambda seg: 1,
    eval_g=lambda comp: 0 if comp == comp_b1 else 1
)

# Recompute direct kernel with Eg: only Path b2 passes!
z_direct_broken = float(np.exp(-1.4))  # Path b1 fails Eg, contributes 0

# The composed sum still thinks Path b1 passes because local checkers both say 1:
# z_composed_sum is unchanged!
r_eval = EvaluatedPathKernel.compute_action_evaluator_residual([comp_b1, comp_b2], eval_set_broken, beta=1.0)
difference = z_direct_broken - z_composed_sum

print(f"Direct Kernel Mass (Broken G3)      : {z_direct_broken:.8f}")
print(f"Composed Prediction (from locals)   : {z_composed_sum:.8f}")
print(f"Direct - Composed Difference        : {difference:.8f}")
print(f"Computed Evaluator Residual R_Eval   : {r_eval:.8f}")

# The identity Z_direct - Z_composed == R_Eval is EXACT
assert abs(difference - r_eval) < 1e-15
print("\n[VERIFIED ON FIXTURE] The Evaluator Residual R_Eval exactly accounts for the composition failure!")


In [ ]:
# Visualizing the Workflow Bipartite Decomposition
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)

# Node coordinates
pos = {
    s_a: (0.1, 0.5),
    s_b1: (0.5, 0.8),
    s_b2: (0.5, 0.2),
    s_c: (0.9, 0.5)
}

# Draw states
for state, (x, y) in pos.items():
    color = '#2b5c8f' if state in [s_a, s_c] else '#e66101'
    ax.scatter(x, y, s=1600, color=color, zorder=4, edgecolor='black', linewidth=1.5)
    ax.text(x, y, state.label, ha='center', va='center', color='white', fontweight='bold', fontsize=9, zorder=5)

# Draw edges (segments)
# Path 1 (Top): Sa -> Sb1 -> Sc
ax.annotate("", xy=pos[s_b1], xytext=pos[s_a], arrowprops=dict(arrowstyle="->", color="black", lw=2))
ax.annotate("", xy=pos[s_c], xytext=pos[s_b1], arrowprops=dict(arrowstyle="->", color="red", lw=2, linestyle="--"))
ax.text(0.3, 0.72, r"$\mathcal{A}_1=0.4$ ($E_1=1$)", fontsize=9, fontweight='bold')
ax.text(0.7, 0.72, r"$\mathcal{A}_2=0.5$ ($E_2=1, E_g=0$ [NGD!])", fontsize=9, color="red", fontweight='bold')

# Path 2 (Bottom): Sa -> Sb2 -> Sc
ax.annotate("", xy=pos[s_b2], xytext=pos[s_a], arrowprops=dict(arrowstyle="->", color="black", lw=2))
ax.annotate("", xy=pos[s_c], xytext=pos[s_b2], arrowprops=dict(arrowstyle="->", color="green", lw=2))
ax.text(0.3, 0.28, r"$\mathcal{A}_1=0.6$ ($E_1=1$)", fontsize=9, fontweight='bold')
ax.text(0.7, 0.28, r"$\mathcal{A}_2=0.8$ ($E_2=1, E_g=1$ [ZDR])", fontsize=9, color="green", fontweight='bold')

# Boundary slice box
rect = plt.Rectangle((0.42, 0.05), 0.16, 0.9, fill=False, edgecolor='#e66101', linestyle=':', linewidth=2)
ax.add_patch(rect)
ax.text(0.5, 0.97, r"Boundary Slice $\mathsf{B} = \{S_{b1}, S_{b2}\}$", ha='center', color='#e66101', fontweight='bold', fontsize=10)

ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.05)
ax.axis('off')
ax.set_title("Agent Trajectories Decomposing Across Boundary Slice B", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Mandatory Claim Hygiene: CAN-Show vs CANNOT-Show

### What this model CAN show:
- Shows that under conditions G1–G3, evaluated path kernels compose with zero residual error ($R_{\mathrm{Eval}} = 0$).
- Shows that when G3 fails, the composition gap $Z_{\mathrm{direct}} - Z_{\mathrm{composed}}$ equals $R_{\mathrm{Eval}} = \sum \Delta_E e^{-\beta \mathcal{A}}$ identically.
- Shows how intermediate checkpoints (boundary slices) enable modular auditing of long-horizon tasks.

### What this model CANNOT show:
- Does **not** show that G1–G3 are *necessary* for zero residual: aggregate cancellation ($R_{\mathrm{Eval}} = 0$) can occur even when G3 fails pointwise! (See Notebook 05).
- Does **not** show that real LLMs decompose uniquely into independent steps (prompt leakage, attention bleed, and hidden state correlations can violate G1).


## 6. Student Exercises

### Exercise 1: The Effect of Action Penalty ($\beta$)
In the broken G3 code cell above, we used $\beta = 1.0$.
What happens to the relative composition error as $\beta \to 0$ (all paths count equally) versus as $\beta \to \infty$ (only the cheapest path matters)?

<details>
<summary><b>Click to see the solution for Exercise 1</b></summary>

- As $\beta \to 0$:  
  $e^0 = 1$. The residual becomes the raw defect sum: $R_{\mathrm{Eval}} = \sum \Delta_E = -1 + 0 = -1$.
- As $\beta \to \infty$:  
  The cheapest path ($\mathcal{A}_1 + \mathcal{A}_2 = 0.9$, Path b1) completely dominates the partition function. Here $Z_{\mathrm{direct}} = e^{-1.4\beta}$, $Z_{\mathrm{composed}} = e^{-0.9\beta} + e^{-1.4\beta}$, and $|R_E| = e^{-0.9\beta}$.
  - Normalized by composed mass: $|R_E| / Z_{\mathrm{composed}} = 1 / (1 + e^{-0.5\beta}) \to 1$ (approaching 100% of composed mass).
  - Normalized by direct mass: $|R_E| / Z_{\mathrm{direct}} = e^{0.5\beta} \to \infty$ (unbounded relative error).
  - Normalized in code: $\mathrm{nRKE} = |R_E| / (Z_{\mathrm{direct}} + \varepsilon) \to 0$ due to the fixed positive $\varepsilon$ floor as both mass terms vanish.
</details>

### Exercise 2: Violating Condition G1
What would happen if Path b1 could exit at *two* different states in $\mathsf{B}$ at the same time? Why does non-unique decomposition break matrix multiplication?

<details>
<summary><b>Click to see the solution for Exercise 2</b></summary>

- If a path decomposes through multiple states in $\mathsf{B}$, it gets counted multiple times in the sum $\sum_{S_b \in \mathsf{B}} Z(S_c \mid S_b) Z(S_b \mid S_a)$.
- This introduces double-counting error, so $Z_{\mathrm{composed}} > Z_{\mathrm{direct}}$ even if evaluators factorize perfectly ($E_g = E_1 E_2$).
- Condition G1 (unique decomposition and Cartesian completeness) structurally guarantees that global sums reindex into product sums. While accidental aggregate cancellation could preserve $Z_{\mathrm{composed}} = Z_{\mathrm{direct}}$ if the duplicated paths carry zero accepted weight, G1 is the required structural foundation for path composition.
</details>


---
## Anchors & Provenance

- **Paper Sections:** §2 (Evaluated Path Kernels), §3 (Kernel Composition Theorem), §4 (Evaluator Gluing Defect and Residual).
- **Claim Ledger IDs:** B4 (EPK definition), C1 (Composition Theorem), C2 (G1-G3 definitions), D2 (Defect-corrected kernel).
- **Code Functions:** `epk_core.BoundarySlice.validate_unique_decomposition`, `epk_core.EvaluatedPathKernel.compute_action_kernel`, `epk_core.EvaluatedPathKernel.compute_action_evaluator_residual`.
- **Lean 4 Mechanization:**
  - `PaperAE004.C1_kernel_composition_under_G1_G2_G3` in `lean4/lggr_i_evaluated_path_kernels.lean`
  - `PaperAE004.L020_03_KernelComposition.lean`
  - `PaperAE004.L020_03b_ProductOfSums.lean`
- **Next Step:** Open [`04_why_green_evaluators_can_lie.ipynb`](04_why_green_evaluators_can_lie.ipynb) to learn about Detector Adequacy and how degenerate evaluators can trick dashboards into reporting false green.
